In [1]:
# torch is the fundamental numerical computing library
# It gives us the 'tensor' object — the GPU/CPU-aware equivalent of a numpy array
# Every weight matrix in the model is a torch.Tensor
import torch

# transformers is HuggingFace's library
# AutoModelForCausalLM: a smart loader that reads config.json,
#   figures out the right model class (Phi3ForCausalLM in our case),
#   builds the module tree, and fills it with weights from the .safetensors file
# AutoTokenizer: loads the vocabulary and handles text → token ID conversion
from transformers import AutoModelForCausalLM, AutoTokenizer

/home/codespace/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_id = "microsoft/Phi-4-mini-instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True   # Phi-4-mini uses custom code in the repo, this allows it to run
)

print("Loading model... (this will take 1-3 minutes and ~8GB RAM)")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,   # load weights in BF16, matching the original format
    device_map="cpu",              # explicitly put everything on CPU — no GPU assumed
    trust_remote_code=True
)

print("Model loaded.")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")


Loading tokenizer...
Loading model... (this will take 1-3 minutes and ~8GB RAM)


Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.88s/it]


Model loaded.
Total parameters: 3,836,021,760


In [3]:
# model.model  →  the Phi3Model (inner body, excluding lm_head)
# .layers      →  the ModuleList of 32 DecoderLayers
# [0]          →  Layer 0 specifically (index 0 of the list)
# .self_attn   →  the Phi3Attention sub-module inside that layer
# .qkv_proj   →  the Linear layer (contains .weight and optionally .bias)
# .weight      →  the actual tensor of numbers — shape [5120, 3072]
layer0_qkv = model.model.layers[0].self_attn.qkv_proj.weight

print(f"Type:   {type(layer0_qkv)}")
print(f"Shape:  {layer0_qkv.shape}")
print(f"Dtype:  {layer0_qkv.dtype}")
print(f"Device: {layer0_qkv.device}")


# .detach() — creates a copy disconnected from PyTorch's computation graph
#   We do this because we're going to do manual math on this tensor
#   Without detach(), PyTorch would try to track every operation for gradient computation
#   which wastes memory and isn't needed since we're not training
#
# .float() — converts from BF16 to FP32 for our inspection math
#   BF16 arithmetic can have small rounding surprises during computation
#   FP32 gives us clean, predictable numbers for analysis
#   We are NOT changing the model's stored weights — this is a local copy
w = layer0_qkv.detach().float()

# These four numbers are your baseline — you will reference them constantly
print(f"Min value:       {w.min().item():.6f}")
print(f"Max value:       {w.max().item():.6f}")
print(f"Max absolute:    {w.abs().max().item():.6f}")
print(f"Mean absolute:   {w.abs().mean().item():.6f}")

Type:   <class 'torch.nn.parameter.Parameter'>
Shape:  torch.Size([5120, 3072])
Dtype:  torch.bfloat16
Device: cpu
Min value:       -1.593750
Max value:       1.460938
Max absolute:    1.593750
Mean absolute:   0.026782


In [4]:
# Same histogram, but sorted to show the most populated buckets first
# This lets us see where the bulk of values actually live
hist = torch.histc(w, bins=50, min=w.min().item(), max=w.max().item())
bucket_width = (w.max().item() - w.min().item()) / 50

buckets = []
for i, count in enumerate(hist):
    bucket_left = w.min().item() + i * bucket_width
    count_int = int(count.item())
    buckets.append((count_int, bucket_left, bucket_left + bucket_width))

# Sort by count descending, show top 15
buckets.sort(reverse=True)
print("\nTop 15 most populated buckets:")
print(f"{'Range':>25}  {'Count':>10}  {'% of total':>10}")
print("-" * 60)
total = w.numel()
for count, left, right in buckets[:15]:
    pct = 100.0 * count / total
    print(f"  [{left:+.3f} to {right:+.3f}]  {count:>10,}  {pct:>9.2f}%")

# Also print these specific summary stats
print(f"\nTotal values in matrix: {total:,}")
print(f"Values between -0.1 and +0.1: {((w > -0.1) & (w < 0.1)).sum().item():,}")
print(f"That is: {100.0 * ((w > -0.1) & (w < 0.1)).sum().item() / total:.1f}% of all values")
print(f"Values beyond ±0.5: {((w.abs() > 0.5)).sum().item():,}")
print(f"That is: {100.0 * (w.abs() > 0.5).sum().item() / total:.1f}% of all values")


Top 15 most populated buckets:
                    Range       Count  % of total
------------------------------------------------------------
  [-0.005 to +0.056]   8,227,407      52.31%
  [-0.066 to -0.005]   6,048,842      38.46%
  [+0.056 to +0.117]     774,529       4.92%
  [-0.127 to -0.066]     498,270       3.17%
  [+0.117 to +0.178]      81,201       0.52%
  [-0.189 to -0.127]      59,178       0.38%
  [+0.178 to +0.239]      14,805       0.09%
  [-0.250 to -0.189]      11,252       0.07%
  [+0.239 to +0.300]       4,102       0.03%
  [-0.311 to -0.250]       3,444       0.02%
  [+0.300 to +0.361]       1,408       0.01%
  [-0.372 to -0.311]       1,237       0.01%
  [+0.361 to +0.422]         662       0.00%
  [-0.433 to -0.372]         551       0.00%
  [+0.422 to +0.483]         351       0.00%

Total values in matrix: 15,728,640
Values between -0.1 and +0.1: 15,375,742
That is: 97.8% of all values
Values beyond ±0.5: 1,016
That is: 0.0% of all values


97.8% of 15.7 million values live between -0.1 and +0.1.

Only 1,016 values — out of 15,728,640 — exceed ±0.5.

And yet — those 1,016 values are setting the scale for all 15,728,640. Because naive quantization takes the single largest absolute value (1.593750) and uses it to define the slot size for the entire matrix.



Let's make this concrete with your actual numbers:

Scale (naive, whole matrix) = 1.593750 / 127 = 0.012549 per slot

A value of 0.050 (very typical, inside the fat middle bucket) maps to:
    0.050 / 0.012549 = 3.98 → rounds to slot 4

A value of 0.030 maps to slot 2.
A value of 0.010 maps to slot 1.
A value of 0.005 maps to slot 0.
A value of -0.005 maps to slot 0.

Everything between -0.006 and +0.006 maps to slot 0.




That -0.005 to +0.056 bucket alone contains 8.2 million values — 52% of the entire matrix — and they're all being compressed into roughly slots 0 through 4. Five slots. Out of 127 available.

The 1,016 outlier values beyond ±0.5 are using slots 40 through 127. Eighty-seven slots. For 0.006% of values.

This is not a small inefficiency. This is the entire problem you are solving.

so how do I visualise the matrix  as rows and columns  I mean 
 [-0.005 to +0.056]   8,227,407      52.31%   [-0.066 to -0.005]   6,048,842      38.46%
these are just the ranges and the valuecounts  and percentages lying in the ranges 

there is no insight on how the weights are placesd in the matrix  how I I know the trend whether the weights are outliers in a trend of columns or rows  I mean how do I imagine the  matrix positional trends I'll need that insight   to take the decision of whether row scaling or column scaling is better?

why did the company even release such a defective model its pretrained yet has outliers even before  the tokens are passed how can it ever give good response

On the "defective model" question — this is a fundamental misconception worth correcting properly.

The model is not defective. It gives excellent responses in BF16. The outliers are not bugs — they are a direct consequence of the model learning to be good.

Here's what actually happens during training. The model is shown trillions of examples. At some point, training discovered that putting a very large weight value in a specific channel of a specific layer dramatically reduced prediction error. Gradient descent — the optimization process — has no constraint saying "keep all weights small." It only cares about one thing: minimize the loss. If a large weight value helps, it stays large. The model doesn't know or care that you'll try to quantize it later.

The outliers you're seeing are features, not bugs. They represent something the model learned was important — a specific direction in weight space that carries disproportionate information. This is why naive quantization degrades quality: you're not rounding unimportant noise, you're rounding something the model specifically learned to make large because it mattered.

This is also why AWQ exists. It was invented precisely because researchers noticed that post-trained models universally have this property — not just Phi-4-mini, but GPT, LLaMA, Mistral, every model above ~6.7B parameters. It's not a Microsoft problem. It's a property of how large neural networks train.

In [5]:
# w is shape [5120, 3072]
# axis=1 means "compute across the 3072 columns, keeping rows separate"
# So row_max_abs[i] = the largest absolute value anywhere in row i
# Result shape: [5120] — one number per row
row_max_abs = w.abs().max(dim=1).values

# Same idea but for columns
# axis=0 means "compute across the 5120 rows, keeping columns separate"  
# col_max_abs[j] = the largest absolute value anywhere in column j
# Result shape: [3072] — one number per column
col_max_abs = w.abs().max(dim=0).values

print("=== ROW ANALYSIS (5120 rows) ===")
print(f"Median row max-abs:  {row_max_abs.median().item():.6f}")
print(f"Mean row max-abs:    {row_max_abs.mean().item():.6f}")
print(f"Max row max-abs:     {row_max_abs.max().item():.6f}")
print(f"Rows where max-abs > 0.5:  {(row_max_abs > 0.5).sum().item()}")
print(f"Rows where max-abs > 1.0:  {(row_max_abs > 1.0).sum().item()}")

print("\n=== COLUMN ANALYSIS (3072 columns) ===")
print(f"Median col max-abs:  {col_max_abs.median().item():.6f}")
print(f"Mean col max-abs:    {col_max_abs.mean().item():.6f}")
print(f"Max col max-abs:     {col_max_abs.max().item():.6f}")
print(f"Cols where max-abs > 0.5:  {(col_max_abs > 0.5).sum().item()}")
print(f"Cols where max-abs > 1.0:  {(col_max_abs > 1.0).sum().item()}")

print("\n=== TOP 10 WORST ROWS (highest max-abs) ===")
top_rows = row_max_abs.topk(10)
for rank, (val, idx) in enumerate(zip(top_rows.values, top_rows.indices)):
    print(f"  Rank {rank+1}: Row {idx.item():>5}  max-abs = {val.item():.6f}")

print("\n=== TOP 10 WORST COLUMNS (highest max-abs) ===")
top_cols = col_max_abs.topk(10)
for rank, (val, idx) in enumerate(zip(top_cols.values, top_cols.indices)):
    print(f"  Rank {rank+1}: Col {idx.item():>5}  max-abs = {val.item():.6f}")

=== ROW ANALYSIS (5120 rows) ===
Median row max-abs:  0.163086
Mean row max-abs:    0.201939
Max row max-abs:     1.593750
Rows where max-abs > 0.5:  195
Rows where max-abs > 1.0:  20

=== COLUMN ANALYSIS (3072 columns) ===
Median col max-abs:  0.231445
Mean col max-abs:    0.252314
Max col max-abs:     1.593750
Cols where max-abs > 0.5:  61
Cols where max-abs > 1.0:  16

=== TOP 10 WORST ROWS (highest max-abs) ===
  Rank 1: Row  2535  max-abs = 1.593750
  Rank 2: Row  2517  max-abs = 1.398438
  Rank 3: Row  3376  max-abs = 1.375000
  Rank 4: Row  2528  max-abs = 1.320312
  Rank 5: Row  1343  max-abs = 1.273438
  Rank 6: Row  2559  max-abs = 1.265625
  Rank 7: Row  4055  max-abs = 1.218750
  Rank 8: Row  2538  max-abs = 1.218750
  Rank 9: Row  3798  max-abs = 1.210938
  Rank 10: Row  3827  max-abs = 1.203125

=== TOP 10 WORST COLUMNS (highest max-abs) ===
  Rank 1: Col  1541  max-abs = 1.593750
  Rank 2: Col  2412  max-abs = 1.460938
  Rank 3: Col  2293  max-abs = 1.429688
  Rank 4: Co